# DE-LU Day-Ahead Futures Hackathon

This notebook walks through the official naive-copy baseline for the student challenge.

## 1. Challenge Objective

Build a strategy that chooses `+1`, `-1`, or `None` each day using only daily information available at decision time.

## 2. Set Up the Notebook Environment

In [ ]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the repository root from this notebook.")

REPO_ROOT = find_repo_root()
SRC_PATH = REPO_ROOT / "src"

for path in (REPO_ROOT, SRC_PATH):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

REPO_ROOT

## 3. Load the Daily Dataset

In [ ]:
import pandas as pd

from energy_modelling.challenge.data import write_challenge_data

data_path = REPO_ROOT / "data" / "challenge" / "daily_public.csv"
if not data_path.exists():
    source_dataset = REPO_ROOT / "kaggle_upload" / "dataset_de_lu.csv"
    if not source_dataset.exists():
        raise FileNotFoundError(
            f"Could not find {data_path} or the source dataset at {source_dataset}."
        )
    write_challenge_data(source_dataset, REPO_ROOT / "data" / "challenge")

daily = pd.read_csv(data_path, parse_dates=["delivery_date"])
daily["delivery_date"] = daily["delivery_date"].dt.date
daily.head()

## 4. Understand the Timing Contract

The daily table mixes two safe feature groups:

- realised features such as actual load, weather, realised price stats, and neighbour prices are lagged by one day
- day-ahead forecast features such as `load_forecast_mw_mean` and `forecast_*` stay aligned to the delivery day because they are assumed to be known before trading

In [ ]:
daily[[
    "delivery_date",
    "load_actual_mw_mean",
    "load_forecast_mw_mean",
    "forecast_solar_mw_mean",
    "weather_temperature_2m_degc_mean",
]].head()

## 5. Inspect the Feature Timing Glossary

In [ ]:
glossary_path = REPO_ROOT / "data" / "challenge" / "daily_public_glossary.csv"
glossary = pd.read_csv(glossary_path)
glossary.groupby("timing_group").size().rename("columns")

In [ ]:
glossary[glossary["timing_group"].isin(["lagged_realised", "same_day_forecast"])].head(12)

## 6. Understand the Split and Target

In [ ]:
daily[["split", "last_settlement_price", "settlement_price", "target_direction"]].head()

In [ ]:
daily["split"].value_counts().sort_index()

## 7. Create the Public Train and Validation Sets

In [ ]:
train = daily[daily["split"] == "train"].copy()
validation = daily[daily["split"] == "validation"].copy()
len(train), len(validation)

## 8. Build the Naive Copy Baseline

The naive baseline always goes long.

In [ ]:
class NaiveCopyStrategy:
    def fit(self, train_data: pd.DataFrame) -> None:
        self.training_rows = len(train_data)

    def act(self, state) -> int | None:
        return 1

    def reset(self) -> None:
        pass

## 9. Backtest the Baseline on 2024

In [ ]:
from datetime import date

from energy_modelling.challenge.runner import run_challenge_backtest

strategy = NaiveCopyStrategy()
result = run_challenge_backtest(
    strategy=strategy,
    daily_data=daily,
    training_end=date(2023, 12, 31),
    evaluation_start=date(2024, 1, 1),
    evaluation_end=date(2024, 12, 31),
)
result.metrics

## 10. Compare Against the Refreshed Public Leaderboard

These are the top strategies on the current 2024 public validation set after fixing forecast timing.

In [ ]:
public_leaderboard = pd.DataFrame([
    {"strategy": "Tiny ML", "total_pnl": 170908.41, "sharpe": 12.682, "max_drawdown": 3092.60},
    {"strategy": "Price Level Mean Reversion", "total_pnl": 79576.68, "sharpe": 4.800, "max_drawdown": 7772.50},
    {"strategy": "Wind Forecast Contrarian", "total_pnl": 56541.80, "sharpe": 3.336, "max_drawdown": 5225.90},
    {"strategy": "Load Forecast Median", "total_pnl": 31466.54, "sharpe": 1.829, "max_drawdown": 5225.90},
    {"strategy": "DEFrance Spread", "total_pnl": 17695.72, "sharpe": 1.024, "max_drawdown": 7842.00},
])
public_leaderboard

## 11. Inspect Daily Predictions and PnL

In [ ]:
pd.DataFrame({
    "prediction": result.predictions,
    "daily_pnl": result.daily_pnl,
    "cumulative_pnl": result.cumulative_pnl,
}).head()

## 12. Turn the Baseline into a Submission Class

Copy the same logic into `submission/student_strategy.py`, then replace the naive decision rule with your own model or heuristic.

In [ ]:
from submission.student_strategy import StudentStrategy

submission_strategy = StudentStrategy()
submission_strategy.fit(train)
submission_strategy.training_rows

## 13. Ideas for Improvement

Try momentum, mean reversion, or a simple classifier trained on both lagged realised features and same-day day-ahead forecast features.